# 01 — Preprocessing v4

Notebook này chạy preprocessing champion v4 theo `spec_v4`.

- Input: `data/raw`
- Output mới: `data/processed_v4_rgb248_r4_exact`
- Manifest: `data/processed_v4_rgb248_r4_exact/manifest.csv`
- Audit run: `audit_output/validation/spec_v4_20260319/preprocessing_run_v4_rgb248_r4_exact`
- Core contract: decode canonical, EXIF orientation, RGB/RGBA only, exact crop `248@4`, không pad/resize/JPEG bottleneck.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    DEFAULT_CONFIG,
    run_pipeline,
    save_manifest,
    scan_dataset_tree,
    summarise_results,
)

RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed_v4_rgb248_r4_exact'
MANIFEST_PATH = PROCESSED_ROOT / 'manifest.csv'
RUN_AUDIT_ROOT = PROJECT_ROOT / 'audit_output' / 'validation' / 'spec_v4_20260319' / 'preprocessing_run_v4_rgb248_r4_exact'
SUMMARY_PATH = RUN_AUDIT_ROOT / 'preprocessing_run_summary.json'
WORKERS = min(12, os.cpu_count() or 8)
FORCE_RERUN = False
SHOW_PROGRESS = False
CONFIG = DEFAULT_CONFIG
RUN_AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

print({'raw_root': str(RAW_ROOT), 'processed_root': str(PROCESSED_ROOT), 'manifest': str(MANIFEST_PATH), 'workers': WORKERS, 'preprocess_version': CONFIG.preprocess_version})

{'raw_root': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\raw', 'processed_root': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact', 'manifest': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact\\manifest.csv', 'workers': 12, 'preprocess_version': 'v4_rgb248_r4_exact'}


## 1. Scan raw dataset

Quét cấu trúc dữ liệu đầu vào trước khi chạy pipeline để khóa số lượng và cấu trúc thư mục.

In [2]:
dataset_scan = scan_dataset_tree(RAW_ROOT)
{
    'root': dataset_scan['root'],
    'total_images': dataset_scan['total_images'],
    'n_subdirs': len(dataset_scan['by_subdir']),
    'sample_subdirs': dict(list(dataset_scan['by_subdir'].items())[:10]),
}

{'root': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\raw',
 'total_images': 87971,
 'n_subdirs': 14,
 'sample_subdirs': {'ADM\\ai': 6000,
  'ADM\\nature': 5997,
  'GLIDE\\ai': 6000,
  'GLIDE\\nature': 5995,
  'Midjourney\\ai': 6000,
  'Midjourney\\nature': 6000,
  'SDv14\\ai': 6000,
  'SDv14\\nature': 5996,
  'SDv15\\ai': 7992,
  'SDv15\\nature': 7994}}

## 2. Run preprocessing v4

Nếu output mới chưa tồn tại, notebook sẽ chạy full preprocessing. Nếu manifest đã có và `FORCE_RERUN=False`, notebook chỉ load lại kết quả.

In [3]:
def manifest_summary(manifest: pd.DataFrame) -> dict[str, int]:
    status_counts = manifest['status'].value_counts().to_dict()
    return {
        'total_scanned': int(len(manifest)),
        'accepted': int(status_counts.get('ACCEPTED', 0)),
        'low_support': int(status_counts.get('LOW_SUPPORT', 0)),
        'unsupported_input': int(status_counts.get('UNSUPPORTED_INPUT', 0)),
        'decode_error': int(status_counts.get('DECODE_ERROR', 0)),
        'saved_patch': int(pd.to_numeric(manifest.get('saved_patch', pd.Series(dtype=int)), errors='coerce').fillna(0).astype(int).sum()),
        'stale_output_removed': int(pd.to_numeric(manifest.get('stale_output_removed', pd.Series(dtype=int)), errors='coerce').fillna(0).astype(int).sum()),
        'orientation_applied': int(pd.to_numeric(manifest.get('orientation_applied', pd.Series(dtype=int)), errors='coerce').fillna(0).astype(int).sum()),
        'alpha_composited': int(pd.to_numeric(manifest.get('alpha_composited', pd.Series(dtype=int)), errors='coerce').fillna(0).astype(int).sum()),
    }

if FORCE_RERUN and PROCESSED_ROOT.exists():
    shutil.rmtree(PROCESSED_ROOT)

if MANIFEST_PATH.exists() and not FORCE_RERUN:
    manifest = pd.read_csv(MANIFEST_PATH)
    run_summary = manifest_summary(manifest)
    run_summary['run_mode'] = 'load_existing_manifest'
else:
    results = run_pipeline(
        RAW_ROOT,
        PROCESSED_ROOT,
        config=CONFIG,
        workers=WORKERS,
        overwrite=True,
        show_progress=SHOW_PROGRESS,
        log_failures=False,
    )
    save_manifest(results, MANIFEST_PATH)
    manifest = pd.read_csv(MANIFEST_PATH)
    run_summary = summarise_results(results)
    run_summary['run_mode'] = 'fresh_pipeline_run'

run_summary.update({
    'raw_root': str(RAW_ROOT),
    'processed_root': str(PROCESSED_ROOT),
    'manifest_path': str(MANIFEST_PATH),
    'workers': WORKERS,
    'preprocess_version': CONFIG.preprocess_version,
    'crop_size': CONFIG.crop_size,
    'residue': [CONFIG.residue_x, CONFIG.residue_y],
    'support_threshold': CONFIG.support_threshold,
})
SUMMARY_PATH.write_text(json.dumps(run_summary, indent=2, ensure_ascii=False), encoding='utf-8')
run_summary

{'total_scanned': 87971,
 'accepted': 85615,
 'low_support': 1573,
 'unsupported_input': 783,
 'decode_error': 0,
 'saved_patch': 85615,
 'stale_output_removed': 0,
 'orientation_applied': 6,
 'alpha_composited': 6000,
 'run_mode': 'load_existing_manifest',
 'raw_root': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\raw',
 'processed_root': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact',
 'manifest_path': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact\\manifest.csv',
 'workers': 12,
 'preprocess_version': 'v4_rgb248_r4_exact',
 'crop_size': 248,
 'residue': [4, 4],
 'support_threshold': 252}

## 3. Post-run sanity checks

Kiểm tra phân bố trạng thái, coverage theo label, mode input và vài hàng manifest đầu tiên.

In [4]:
status_by_label = manifest.groupby(['label', 'status']).size().unstack(fill_value=0)
status_by_label

status,ACCEPTED,LOW_SUPPORT,UNSUPPORTED_INPUT
label,,,
ai,43992,0,0
nature,41623,1573,783


In [5]:
manifest[['status', 'input_format', 'input_mode', 'normalized_mode', 'support', 'crop_origin_x', 'crop_origin_y', 'saved_patch']].head(12)

,status,input_format,input_mode,normalized_mode,support,crop_origin_x,crop_origin_y,saved_patch
0,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
1,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
2,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
3,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
4,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
5,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
6,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
7,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
8,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True
9,ACCEPTED,PNG,RGBA,RGB,256,4.0,4.0,True


In [6]:
accepted = manifest.loc[manifest['status'] == 'ACCEPTED'].copy()
accepted[['label', 'generator', 'input_mode', 'output_path']].head(10)

,label,generator,input_mode,output_path
0,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
1,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
2,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
3,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
4,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
5,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
6,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
7,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
8,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
9,ai,ADM,RGBA,C:\Users\USER\Desktop\ai_detector_img\data\pro...
